In [14]:
import pandas as pd
import numpy as np
import os


In [15]:
val_path = "val_templatetest_transformed.csv"
feat_imp_path = "models/xgb-kfold-binary-13/feature_importance.csv"
output_path = "/mnt/obj/val/val_templatetest_transform.csv"

In [16]:
df = pd.read_csv(val_path)
feature_importance = pd.read_csv(feat_imp_path)

In [17]:
df.shape

(22607, 70)

In [18]:
top5_features = feature_importance.sort_values(by='importance', ascending=False).head(5)['feature'].tolist()
print("Top 5 Features:", top5_features)

Top 5 Features: ['sub_grade', 'open_rv_12m', 'int_rate', 'verification_status_Not Verified', 'funded_amnt']


In [19]:
# === Identify features and label ===
label_col = 'risk_level'
feature_cols = [col for col in df.columns if col != label_col]
non_top_features = [f for f in feature_cols if f not in top5_features]

In [20]:
# === Create modified datasets ===

# 1. Non-top features ramped
df1 = df.copy()
df1[non_top_features] = df1[non_top_features] + 10_000_000
# df1["variation"] = "non_top_ramped"
df1['risk_level']=df['risk_level']
df1.to_csv("non_top_upscaled.csv")

# 2. Top 5 features ramped
df1 = df.copy()
df1[top5_features] = df1[top5_features] + 10_000_000
# df1["variation"] = "top_ramped"
df1['risk_level']=df['risk_level']
df1.to_csv("top_upscaled.csv")

# 3. Non-top features zeroed
df1 = df.copy()
df1[non_top_features] = 0
# df1["variation"] = "non_top_zero"
df1['risk_level']=df['risk_level']
df1.to_csv("non_top_downscaled.csv")

# 4. Top 5 features zeroed
df1 = df.copy()
df1[top5_features] = 0
# df1["variation"] = "top_zero"
df1['risk_level']=df['risk_level']
df1.to_csv("top_downscaled.csv")

# 5. Add random noise to all features (except label)
df1 = df.copy()
noise = np.random.normal(loc=0.0, scale=1.0, size=df1[feature_cols].shape)
df1[feature_cols] = df1[feature_cols] + noise
# df1["variation"] = "noise_added"
df1['risk_level']=df['risk_level']
df1.to_csv("random_noise.csv")

# # 0. Original with marker
# df_original = df.copy()
# df_original["variation"] = "original"

In [ ]:
# === Concatenate and save ===
final_df = pd.concat([df_original, df1, df2, df3, df4, df5], ignore_index=True)
os.makedirs(os.path.dirname(output_path), exist_ok=True)
final_df.to_csv(output_path, index=False)

print(f"✅ Transformed dataset saved to: {output_path}")